# Literature Analysis Table for a Critical Scoping Review of the BBNJ Clearing-House Mechanism at the Benguela Current EEZ-ABNJ Interface Exploration with `mlcroissant`
This notebook guides the exploration of a FAIR^2 literature analysis table concerning marine biodiversity governance at the Benguela Current EEZ-ABNJ interface. We'll use the `mlcroissant` library to load and process the dataset, referencing all entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.pe47-9ata/fair2.json

In [ ]:
# Install `mlcroissant` if it is not already available
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.pe47-9ata/fair2.json"

# Load Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata without subscripting
metadata = dataset.metadata

# Print basic dataset information
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

We'll enumerate the dataset's record sets, referencing their `@id` attributes. Then, for each record set, we list the available fields and columns with their `@id`s.

In [ ]:
# List record sets and show their fields/columns by @id
record_sets = dataset.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs.id}")

# Show fields (schema) for each record set by @id
for rs in record_sets:
    print(f"\nFields for RecordSet '{rs.name}' (@id={rs.id}):")
    for field in rs.fields:
        print(f"  Field name: {field.name}, @id: {field.id}, Type: {field.data_type}")
    if hasattr(rs, 'columns'):
        print(f"Columns for RecordSet '{rs.name}' (@id={rs.id}):")
        for col in rs.columns:
            print(f"    Column name: {col.name}, @id: {col.id}, Type: {col.data_type}")

## 3. Data Extraction
Load data from the main record set(s) into Pandas DataFrames. All access is referenced using the entity `@id`s found above.

We'll extract all available record sets for analysis.

In [ ]:
# Prepare a dataframe for each record set using their @id
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nDataFrame for RecordSet @id '{rs_id}':")
    print(df.columns.tolist())
    print(df.head(2))

# Pick primary record set to use for EDA
primary_rs_id = record_set_ids[0]
primary_df = dataframes[primary_rs_id]

# List its columns
print(f"\nColumns in primary record set (@id={primary_rs_id}): {primary_df.columns.tolist()}")
primary_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps:
- Filtering by a numeric field (referenced by `@id`)
- Normalization
- Grouping/categorization

We'll search for a numeric field by its `@id` and demonstrate filtering and normalization.

In [ ]:
# Identify a numeric field by @id
# List numeric fields
numeric_fields = []
primary_record_set = next((rs for rs in record_sets if rs.id == primary_rs_id), None)
if primary_record_set:
    for field in primary_record_set.fields:
        if field.data_type in ["Float", "Integer", "Number"]:  # common numeric types
            numeric_fields.append(field.id)

if not numeric_fields:
    print("No numeric fields found in the primary record set.")
else:
    print("Numeric fields by @id:", numeric_fields)

    # Pick first numeric field for demonstration
    numeric_field_id = numeric_fields[0]

    # Filtering
    threshold = 1  # Change as suitable for this dataset
    try:
        filtered_df = primary_df[primary_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

    except Exception as e:
        print(f"Could not filter or normalize numeric field '{numeric_field_id}':", e)

    # Group by a non-numeric field (if available)
    group_field_id = None
    for field in primary_record_set.fields:
        if field.data_type == "Text":
            group_field_id = field.id
            break

    if group_field_id and group_field_id in filtered_df.columns:
        try:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        except Exception as e:
            print(f"Could not group records by '{group_field_id}':", e)
    else:
        print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We demonstrate visualizing the numeric field's distribution and, if possible, the grouping relationship.

In [ ]:
# If numeric_field_id is defined and exists, plot its distribution
if numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(primary_df[numeric_field_id].dropna(), bins=10)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # Conditional plot: Grouped mean vs group_field_id
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(10,4))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id])
        plt.xticks(rotation=60)
        plt.title(f"Mean of {numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated step-by-step exploration of a FAIR^2 literature analysis dataset concerning marine biodiversity data governance. Using `mlcroissant` and consistently referencing the dataset's schema by `@id`, we loaded metadata, extracted record set contents, provided sample data extraction, performed basic filtering and normalization by field ID, and visualized numeric distributions. These operations can be extended for deeper domain analysis, evidence synthesis, and policy-oriented research.

For further exploration, consult the dataset documentation and source schema for advanced analysis, relationship mapping across fields, and reproducible benchmarking.